# Clase 7 — El loop de simulación

Poner el tiempo en marcha: recorrer los snapshots, enviar órdenes en cada paso y llevar la cuenta de caja, inventario y equity con PositionTracker.

**Hoy construyes:** Market: reproducir snapshots y ejecutar en el tiempo.

## Cómo usar este cuaderno

- **Núcleo (en clase):** ejercicios 1 a 3.
- **Si vamos bien:** ejercicios 4 en adelante.
- **Casa / auxiliares:** el cuaderno `*_auxiliary.ipynb`.

Inténtalo, ejecuta la comprobación (`assert`) y mira la solución solo si te atascas.

## 1. Cuenta los pasos

**Practicas:** el loop step().

Recorre todo el mercado con un while y cuenta los snapshots en `steps`.

In [ ]:
from exchange import Market
m = Market.sample()
steps = 0

In [ ]:
assert steps == 500
print('ok')

### Solución guiada

```python
from exchange import Market
m = Market.sample()
steps = 0
while True:
    book = m.step()
    if book is None: break
    steps += 1
```

## 2. Ejecuta una orden en un paso

**Practicas:** submit.

Avanza un paso y envía una market buy de 0.2 con `m.submit(...)`. Guarda `filled`.

In [ ]:
from exchange import Market, Order, Side, OrderType
m = Market.sample()
m.step()
filled = None

In [ ]:
assert abs(filled - 0.2) < 1e-9
print('ok')

### Solución guiada

```python
fills = m.submit(Order('BTCUSDT', Side.BUY, 0.2, order_type=OrderType.MARKET))
filled = sum(f.size for f in fills)
```

## 3. Acumula posición en el tiempo

**Practicas:** loop + tracker.

Cada 50 pasos compra 0.1 (market). Aplica los fills a un `PositionTracker`. Guarda `final_pos`.

In [ ]:
from exchange import Market, Order, Side, OrderType, PositionTracker
m = Market.sample()
tracker = PositionTracker()
i = 0
final_pos = None

In [ ]:
assert final_pos > 0, 'has comprado varias veces: posición larga'
print('ok  pos=%.2f' % final_pos)

### Solución guiada

```python
i = 0
while True:
    book = m.step()
    if book is None: break
    if i % 50 == 0:
        for f in m.submit(Order('BTCUSDT', Side.BUY, 0.1, order_type=OrderType.MARKET)):
            tracker.apply_fill(f)
    i += 1
final_pos = tracker.position
```

## 4. Equity final

**Practicas:** marcar a mercado.

Continuando, marca el equity al último mid visto. Guarda `equity`.

In [ ]:
from exchange import Market, Order, Side, OrderType, PositionTracker
m = Market.sample()
tracker = PositionTracker()
last_mid = 0
i = 0
while True:
    book = m.step()
    if book is None: break
    last_mid = book.mid
    if i % 50 == 0:
        for f in m.submit(Order('BTCUSDT', Side.BUY, 0.1, order_type=OrderType.MARKET)):
            tracker.apply_fill(f)
    i += 1
equity = None

In [ ]:
assert isinstance(equity, float)
print('ok  equity=%.2f' % equity)

### Solución guiada

```python
equity = tracker.equity(last_mid)
```

## Cierre

Mercado = estado (libro) + dinámica (matching) + tiempo (el loop). Todo junto, ya simulas.

Si llegas al ejercicio 3 ya tienes el núcleo. Los siguientes y los auxiliares consolidan.

**Siguiente clase:** seguimos construyendo el motor sobre esta pieza.